# OKD on CRC: installation and cluster operations

This lab runs the community OKD distribution in a CodeReady Containers (CRC) virtual machine. Run every shell block in a terminal on the Linux VM. Commands that begin with `sudo` require the VM user's sudo access.

## Software and data locations

| Purpose | Location |
|---|---|
| Course deployment files | `$HOME/openshift` |
| Stable administrator kubeconfig | `$HOME/openshift/.state/admin.kubeconfig` |
| CRC program installed by this course | `$HOME/.local/bin/crc` |
| CRC configuration, cache and VM disks | `$HOME/.crc` |
| Active CRC virtual disk | `$HOME/.crc/machines/crc/crc.qcow2` |
| Extracted OKD base image cache | `$HOME/.crc/cache` |
| System libvirt virtual machine | `crc` |
| Big-data Kubernetes project | `bigdata` |

The OKD operating system and cluster data are stored inside the CRC virtual disk. `/opt` on the Linux host is separate from the CRC installation.

## Verify host requirements

CRC uses KVM and the system libvirt service. The login user should belong to the `libvirt` and `kvm` groups. A new group membership becomes active after signing out and signing in again.

In [ ]:
uname -m
egrep -c '(vmx|svm)' /proc/cpuinfo
id
systemctl is-active libvirtd
sudo virsh -c qemu:///system list --all
free -h
df -h /


If the required groups are missing, add the current user and then sign out of the VM completely and sign in again.

In [ ]:
sudo usermod -aG libvirt,kvm "$USER"
getent group libvirt
getent group kvm


## Install and prepare CRC

Install the host packages and CRC directly from this notebook. Run these commands as the ordinary sudo-capable login user. The download is skipped when `$HOME/.local/bin/crc` already exists.

In [ ]:
sudo apt-get update
sudo apt-get install -y qemu-kvm libvirt-daemon-system libvirt-clients network-manager apache2-utils skopeo curl xz-utils python3-boto3
mkdir -p "$HOME/.local/bin" "$HOME/downloads"
if [[ ! -x "$HOME/.local/bin/crc" ]]; then
  curl -fL --retry 2 \
    https://developers.redhat.com/content-gateway/rest/mirror/pub/cgw/crc/2.64.0/crc-linux-amd64.tar.xz \
    -o "$HOME/downloads/crc-linux-amd64.tar.xz"
  tar -xf "$HOME/downloads/crc-linux-amd64.tar.xz" -C "$HOME/downloads"
  find "$HOME/downloads" -name crc -type f -exec cp {} "$HOME/.local/bin/crc" \;
  chmod 755 "$HOME/.local/bin/crc"
fi
export PATH="$HOME/.local/bin:$PATH"
crc version
crc config set consent-telemetry no
crc config set preset okd
crc config set cpus 4
crc config set memory 16384
crc config set disk-size 50
crc setup
crc config view


For the classroom profile, configure four virtual CPUs, 16 GiB of memory, and a 50 GiB CRC disk before the first start. CRC memory is specified in MiB. Increasing the disk setting after creation does not resize an existing virtual disk.

In [ ]:
crc config set preset okd
crc config set consent-telemetry no
crc config set cpus 4
crc config set memory 16384
crc config set disk-size 50
crc config view


## Start OKD and load the command environment

After installing the `.bashrc` functions below, `okd-start` starts CRC and prepares `oc`. The first start can take several minutes.

In [ ]:
export PATH="$HOME/.local/bin:$PATH"
crc start
eval "$(crc oc-env)"
export KUBECONFIG="$HOME/.crc/machines/crc/kubeconfig"
oc whoami
oc get nodes


The `.bashrc` functions below load CRC's `oc` client and copy CRC's current administrator kubeconfig to `$HOME/.kube/crc-admin.kubeconfig`. This keeps terminal initialization independent of the course repository.

## Add the OKD environment command to `.bashrc`

Run this block once. It appends an `okd-env` function only when the marked section is absent. The function loads the course kubeconfig and selects the `bigdata` project. It does not start the cluster automatically.

In [ ]:
touch "$HOME/.bashrc"
if ! grep -q '^# BEGIN OKD LAB$' "$HOME/.bashrc"; then
  tee -a "$HOME/.bashrc" >/dev/null <<'EOF'

# BEGIN OKD LAB
export PATH="$HOME/.local/bin:$PATH"
export KUBECONFIG="$HOME/.kube/crc-admin.kubeconfig"

okd-env() {
    export PATH="$HOME/.local/bin:$PATH"
    local crc_state
    if ! crc_state="$(crc status 2>&1)"; then
        printf '%s\n' "$crc_state" >&2
        echo "Cannot read CRC status. Check libvirt access, then run: okd-start" >&2
        return 1
    fi
    if ! grep -Eq '^OpenShift:[[:space:]]+Running' <<<"$crc_state"; then
        echo "OKD is stopped. Run: okd-start" >&2
        return 1
    fi
    eval "$(crc oc-env)"
    mkdir -p "$HOME/.kube"
    if [[ ! -s "$HOME/.crc/machines/crc/kubeconfig" ]]; then
        echo "CRC kubeconfig was not found. Run: okd-start" >&2
        return 1
    fi
    cp "$HOME/.crc/machines/crc/kubeconfig" "$KUBECONFIG"
    chmod 600 "$KUBECONFIG"
    oc project bigdata 2>/dev/null || oc new-project bigdata
}

okd-start() {
    crc start && okd-env
}

okd-status() {
    crc status
}

okd-stop() {
    crc stop
}

okd-restart() {
    crc stop && crc start && okd-env
}

echo "OKD commands: okd-start, okd-env, okd-status, okd-stop, okd-restart"
# END OKD LAB
EOF
fi
source "$HOME/.bashrc"


Start CRC and load the OKD environment with `okd-start`. In later terminals, use `okd-env` when the cluster is already running. These functions work without changing directory and without sourcing a file from the course repository.

In [ ]:
okd-start
oc whoami
oc project
oc get nodes
oc get pods -n bigdata


`$HOME/.local/bin` is the standard per-user location used here for `crc`. Adding it to the front of `PATH` does not change Java, Hadoop, Hive, Spark, or Kafka installations. A command with the same name in this directory takes precedence over a system command. Check the resolved programs after installation; `crc oc-env` then selects the `oc` client supplied with CRC.

In [ ]:
type -a crc
type -a oc
type -a kubectl
command -v crc
command -v oc
command -v kubectl
crc version
oc version --client
kubectl version --client


In [ ]:
okd-env
printf 'KUBECONFIG=%s\n' "$KUBECONFIG"
oc version
oc cluster-info


## Check cluster status

`crc status` checks the CRC virtual machine. The `oc` commands check the Kubernetes API, nodes, OpenShift cluster operators, projects, and workloads.

In [ ]:
okd-status
okd-env
oc get nodes -o wide
oc get clusterversion
oc get clusteroperators
oc get projects
oc get pods -n bigdata -o wide
oc get deployments,statefulsets,services,pvc -n bigdata


A healthy single-node lab has one `Ready` node. Cluster operators may briefly show `PROGRESSING=True` after startup. Investigate an operator when it remains degraded or unavailable.

In [ ]:
oc describe clusteroperator <operator-name>
oc get events -A --sort-by=.lastTimestamp | tail -50
oc get pods -A | grep -vE 'Running|Completed'


Replace `<operator-name>` with a name from `oc get clusteroperators`, such as `authentication`.

## Stop, start, and restart the whole OKD cluster

Stop CRC cleanly before shutting down or cloning the Linux VM. CRC has start and stop operations; a controlled restart is a stop followed by a start.

In [ ]:
okd-stop
sudo virsh -c qemu:///system domstate crc


In [ ]:
export PATH="$HOME/.local/bin:$PATH"
crc start
eval "$(crc oc-env)"
export KUBECONFIG="$HOME/.crc/machines/crc/kubeconfig"
oc whoami
oc get nodes


For a full restart, run the stop block, wait until `domstate` reports `shut off`, and then run the start block. Do not use `crc delete` for restart: it permanently removes the CRC cluster and its data.

## Manage workloads with `oc`

These operations affect applications inside OKD and do not stop the CRC virtual machine.

In [ ]:
oc -n bigdata get pods
oc -n bigdata get pod <pod-name> -o yaml
oc -n bigdata describe pod <pod-name>
oc -n bigdata logs <pod-name> --all-containers --tail=100
oc -n bigdata exec -it <pod-name> -- bash


In [ ]:
oc -n bigdata rollout restart deployment/<deployment-name>
oc -n bigdata rollout status deployment/<deployment-name> --timeout=5m
oc -n bigdata scale deployment/<deployment-name> --replicas=0
oc -n bigdata scale deployment/<deployment-name> --replicas=1
oc -n bigdata delete pod <pod-name>


Deleting a pod managed by a Deployment or StatefulSet makes its controller create a replacement. Scaling to zero is the normal way to stop a Deployment until it is needed again.

## Use `kubectl` with OKD

OKD exposes a Kubernetes API, so standard `kubectl` commands work with the same kubeconfig. `oc` includes Kubernetes operations and adds OpenShift operations such as projects, routes, image streams, builds, OAuth, and cluster operators. Prefer `oc` throughout this course.

In [ ]:
okd-env
kubectl version --client
kubectl config current-context
kubectl get nodes
kubectl -n bigdata get pods,services
kubectl -n bigdata describe pod <pod-name>
kubectl -n bigdata logs <pod-name> --tail=100
kubectl -n bigdata exec -it <pod-name> -- bash


The namespace selection differs slightly: `oc project bigdata` changes the active project, while portable `kubectl` commands commonly include `-n bigdata`.

## OpenShift-specific resources


In [ ]:
oc project bigdata
oc get routes -n bigdata
oc get imagestreams -n bigdata
oc get securitycontextconstraints
oc auth can-i --list -n bigdata
oc get console.operator.openshift.io cluster -o jsonpath='{.status.consoleURL}{"\n"}'


## Resource and storage checks


In [ ]:
crc config view
du -sh "$HOME/.crc" "$HOME/openshift"
df -h /
oc describe node | sed -n '/Capacity:/,/System Info:/p'
oc get storageclass
oc get pvc -A
oc adm top nodes
oc adm top pods -n bigdata --containers


Metrics may take a short time to become available after cluster startup. The host filesystem must have enough free space for both the CRC virtual disk and container image builds.

## Diagnose CRC access problems

If `crc status` reports that it cannot connect to the KVM driver, confirm group membership and query the CRC domain through system libvirt.

In [ ]:
id
systemctl status libvirtd --no-pager
sudo virsh -c qemu:///system list --all
sudo virsh -c qemu:///system domstate crc
pgrep -af 'qemu-system.*crc' || echo 'No CRC QEMU process'
crc status


When the group list is correct but the current terminal still cannot access libvirt, sign out of the Linux login session and sign in again. If CRC is running but `oc` cannot connect, reload `$HOME/openshift/env.sh` and inspect the API address with `oc config view --minify`.